# Financial Market Regime & Event Intelligence Engine
## Notebook 05: Pretrained Financial Sentiment Analysis (FinBERT)

Welcome to Notebook 05! Building upon our news ingestion pipeline from Notebook 04, we now implement the first NLP layer using **FinBERT**, a state-of-the-art transformer model specifically pre-trained on financial text.

### Objectives:
1. **Reproduce Ingestion Pipeline**: Ingest live financial news headlines using `yfinance` across S&P 500 benchmarks and major market-moving equities.
2. **Pretrained Inference**: Apply the pretrained `ProsusAI/finbert` model to evaluate headline sentiment without model training or fine-tuning.
3. **Enrich Dataset**: Add `sentiment_label` and `sentiment_score` columns while preserving original metadata (`published_at`, `headline`, `publisher`, `query_ticker`, `summary`, `url`).
4. **Sentiment Breakdown & Statistics**: Compute counts and percentage distributions for Positive, Negative, and Neutral headlines.
5. **Interactive Visualizations**: Create Plotly charts displaying overall sentiment distribution and temporal sentiment evolution over time.
6. **Case Study Analysis**: Inspect specific headlines and explain why FinBERT's predictions align with financial domain language.

---
### Step 1: Domain-Specific Financial NLP Explanation

#### 1. What is FinBERT?
- **FinBERT** is a Specialized BERT (Bidirectional Encoder Representations from Transformers) model fine-tuned on financial corpora, including Financial PhraseBank, corporate 10-K/10-Q filings, earnings call transcripts, and market news.

#### 2. Why Financial-Domain LMs Outperform Generic Sentiment Models:
- Generic sentiment tools (e.g., standard VADER, TextBlob, or general BERT) classify words like *"liability"*, *"debt"*, *"risk"*, or *"short"* as negative. In financial contexts, these words are standard terminology.
- Phrases like *"revenue beat estimates but guidance cut"* or *"fed holds rates steady"* carry subtle economic nuances that general-purpose models misinterpret.
- FinBERT understands domain-specific language (e.g., *"bullish"*, *"bearish"*, *"rate cuts"*, *"price target downgrade"*).

#### 3. What the `sentiment_score` Represents:
- The `sentiment_score` is the **softmax probability confidence** ($[0.0, 1.0]$) output by FinBERT's final classification layer for the predicted class label.

#### 4. Why Sentiment Does NOT Automatically Dictate Market Direction:
- Markets operate on **expectations vs. reality**. Positive news that fails to meet elevated market expectations can trigger price sell-offs.
- Market dynamics depend on broader macroeconomic regimes, liquidity conditions, and technical positioning. News sentiment provides qualitative event intelligence to contextualize quantitative regime shifts.

---
### Step 2: Import Required Libraries

**Why we do this:**
- `pandas` & `numpy`: Table manipulation and statistical processing.
- `plotly.express`: Interactive visualization.
- `yfinance`: Ingesting live financial news data.
- `transformers.pipeline`: Hugging Face interface for pretrained transformer inference.
- `torch`: PyTorch execution backend for inference.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import yfinance as yf
import torch
from transformers import pipeline

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 90)
print(f"Libraries imported. PyTorch version: {torch.__version__}")

Libraries imported. PyTorch version: 2.14.0+cpu


---
### Step 3: Ingest & Clean Live Financial News Data

We query news items across 12 key financial tickers, convert timestamps to UTC, remove duplicate headlines, and handle missing fields.

In [2]:
target_tickers = [
    "^GSPC", "SPY", "^VIX", "TLT", "GLD", "QQQ",
    "AAPL", "MSFT", "NVDA", "AMZN", "JPM", "GS"
]

raw_records = []

for ticker in target_tickers:
    try:
        news_items = yf.Ticker(ticker).news
        for item in news_items:
            content = item.get("content", item)
            headline = content.get("title")
            pub_date = content.get("pubDate")
            summary = content.get("summary") or content.get("description", "")
            
            provider_info = content.get("provider", {})
            publisher = provider_info.get("displayName", "Unknown") if isinstance(provider_info, dict) else "Unknown"
            
            canonical_info = content.get("canonicalUrl", {})
            click_info = content.get("clickThroughUrl", {})
            url = canonical_info.get("url") if isinstance(canonical_info, dict) and canonical_info.get("url") else (click_info.get("url", "") if isinstance(click_info, dict) else "")
            
            raw_records.append({
                "query_ticker": ticker,
                "headline": headline,
                "pub_date_raw": pub_date,
                "summary": summary,
                "publisher": publisher,
                "url": url
            })
    except Exception as e:
        print(f"Warning for {ticker}: {e}")

news_df = pd.DataFrame(raw_records)

# Clean and deduplicate
news_df["published_at"] = pd.to_datetime(news_df["pub_date_raw"], utc=True)
news_df = news_df.dropna(subset=["headline"]).drop_duplicates(subset=["headline"]).copy()
news_df["summary"] = news_df["summary"].fillna("N/A")
news_df["publisher"] = news_df["publisher"].fillna("Unknown")
news_df["url"] = news_df["url"].fillna("")

cols = ["published_at", "headline", "publisher", "query_ticker", "summary", "url"]
news_df = news_df[cols].sort_values(by="published_at", ascending=False).reset_index(drop=True)

print(f"Cleaned Financial News Dataset Shape: {news_df.shape}")

Cleaned Financial News Dataset Shape: (108, 6)


---
### Step 4: Load Pretrained FinBERT & Execute Sentiment Inference

We load `ProsusAI/finbert` using Hugging Face's `pipeline` API and perform batch inference on all collected headlines.

In [3]:
print("Loading pretrained FinBERT model ('ProsusAI/finbert')...")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert"
)

headlines_list = news_df["headline"].tolist()
print(f"Running FinBERT sentiment inference on {len(headlines_list)} headlines...")

# Perform model inference
nlp_results = sentiment_pipeline(headlines_list)

# Extract labels and confidence scores
news_df["sentiment_label"] = [res["label"] for res in nlp_results]
news_df["sentiment_score"] = [round(res["score"], 4) for res in nlp_results]

print("\n=== FinBERT Sentiment Inference Complete ===\n")
print("--- Sample of 10 Processed News Records with Sentiment ---")
display(news_df[["published_at", "headline", "sentiment_label", "sentiment_score", "publisher"]].head(10))

Loading pretrained FinBERT model ('ProsusAI/finbert')...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Running FinBERT sentiment inference on 108 headlines...



=== FinBERT Sentiment Inference Complete ===

--- Sample of 10 Processed News Records with Sentiment ---


,published_at,headline,sentiment_label,sentiment_score,publisher
0,2026-09-19 15:35:00+00:00,What S&P 500 Gains of 9.5% in the First Half Signal for the Rest of the Year,positive,0.8831,Motley Fool
1,2026-09-19 15:33:00+00:00,Amazon cargo delivery company files for Chapter 11 bankruptcy,negative,0.8800,TheStreet
2,2026-09-19 15:30:00+00:00,Anthropic IPO: 1 Key Lesson Investors Can Learn From SpaceX,neutral,0.8572,Motley Fool
3,2026-09-19 15:20:00+00:00,"If There's Just 1 Move All Investors Should Make Right Now, History Says It's This",neutral,0.9329,Motley Fool
4,2026-09-19 15:11:11+00:00,"Coinbase Files With CFTC To List US Single-Stock Perpetual Futures, Including AAPL And...",neutral,0.9322,Stocktwits
5,2026-09-19 15:05:00+00:00,The Stock Market's Biggest Companies Are Losing Their Grip. Here's the ETF I'd Buy If ...,neutral,0.6156,Motley Fool
6,2026-09-19 14:50:00+00:00,"Warren Buffett Watched Alphabet's Stock Price Climb 9,000% Before He Decided to Invest...",neutral,0.8828,Motley Fool
7,2026-09-19 14:33:06+00:00,CVS division completes Chapter 11 bankruptcy liquidation,negative,0.8089,TheStreet
8,2026-09-19 14:32:00+00:00,"Upstart, Affirm, and SoFi All Lend to the Same Borrowers. Only One of Them Funds With ...",neutral,0.9521,Motley Fool
9,2026-09-19 14:10:33+00:00,How Investors Are Reacting To Ultragenyx Pharmaceutical (RARE) First-In-Disease Gene T...,neutral,0.8758,Simply Wall St.


---
### Step 5: Sentiment Distribution Summary & Breakdown

We compute total counts and percentage distributions across Positive, Negative, and Neutral sentiment classes.

In [4]:
sentiment_counts = news_df["sentiment_label"].value_counts()
sentiment_percentages = (news_df["sentiment_label"].value_counts(normalize=True) * 100).round(2)

summary_df = pd.DataFrame({
    "Sentiment_Class": sentiment_counts.index,
    "Article_Count": sentiment_counts.values,
    "Percentage": sentiment_percentages.values
})

print("--- FinBERT Sentiment Distribution Summary ---")
display(summary_df)

--- FinBERT Sentiment Distribution Summary ---


,Sentiment_Class,Article_Count,Percentage
0,neutral,54,50.00
1,negative,37,34.26
2,positive,17,15.74


---
### Step 6: Visualize Overall Sentiment Breakdown (Plotly Donut Chart)

In [5]:
fig_pie = px.pie(
    summary_df,
    names="Sentiment_Class",
    values="Article_Count",
    title="Financial News Sentiment Distribution (FinBERT)",
    color="Sentiment_Class",
    color_discrete_map={
        "positive": "#2ca02c",
        "neutral": "#7f7f7f",
        "negative": "#d62728"
    },
    hole=0.4,
    template="plotly_white"
)

fig_pie.update_layout(title_x=0.5)
fig_pie.show()

---
### Step 7: Visualize Sentiment Timeline (Plotly Grouped Bar Chart)

In [6]:
timeline_df = news_df.copy()
timeline_df["date_only"] = timeline_df["published_at"].dt.date

daily_sentiment_df = timeline_df.groupby(["date_only", "sentiment_label"]).size().reset_index(name="count")

fig_sentiment_ts = px.bar(
    daily_sentiment_df,
    x="date_only",
    y="count",
    color="sentiment_label",
    barmode="group",
    title="Financial News Sentiment Timeline by Class",
    labels={"date_only": "Publication Date", "count": "Article Count", "sentiment_label": "Sentiment"},
    color_discrete_map={
        "positive": "#2ca02c",
        "neutral": "#7f7f7f",
        "negative": "#d62728"
    },
    template="plotly_white"
)

fig_sentiment_ts.update_layout(title_x=0.5)
fig_sentiment_ts.show()

---
### Step 8: Headline Case Studies & Financial Language Evaluation

Below are sample headlines extracted from our dataset along with FinBERT's predicted sentiment label and confidence score:

In [7]:
# Select a sample of 4 distinct headlines
sample_case_df = news_df.drop_duplicates(subset=["sentiment_label"]).head(4)
if len(sample_case_df) < 4:
    sample_case_df = news_df.head(4)

display(sample_case_df[["headline", "sentiment_label", "sentiment_score", "publisher"]])

,headline,sentiment_label,sentiment_score,publisher
0,What S&P 500 Gains of 9.5% in the First Half Signal for the Rest of the Year,positive,0.8831,Motley Fool
1,Amazon cargo delivery company files for Chapter 11 bankruptcy,negative,0.8800,TheStreet
2,Anthropic IPO: 1 Key Lesson Investors Can Learn From SpaceX,neutral,0.8572,Motley Fool
3,"If There's Just 1 Move All Investors Should Make Right Now, History Says It's This",neutral,0.9329,Motley Fool


#### Financial Language Analysis:
- **Negative Predictions**: Headlines containing phrases like *"cutting year-end target"*, *"downgrade"*, *"revenue miss"*, or *"inflation resurgence"* correctly trigger `negative` sentiment because FinBERT recognizes price target cuts as bearish signals.
- **Positive Predictions**: Headlines featuring terms like *"rallies"*, *"earnings beat"*, *"cooling inflation"*, or *"upgrade"* are classified as `positive` because they signal improving corporate profitability or favorable macroeconomic conditions.
- **Neutral Predictions**: Informational reporting (e.g., *"Fed announcement scheduled for Tuesday"* or company filing notices) is classified as `neutral` due to the absence of directional bias.